# PRACTICE 2: PREDICTING PRODUCT SALES
## CHẶNG 2: TIỀN XỬ LÝ DỮ LIỆU & KỸ THUẬT ĐẶC TRƯNG (DATA PREPROCESSING & FEATURE ENGINEERING PIPELINE)

> **Mục tiêu kỹ thuật:**
> - Làm sạch dữ liệu thô: lọc đơn Cancelled, đơn âm/null, khử trùng lặp (Zero Duplicate Leakage) -> clean_amazon_sales.csv.
> - Trích xuất 11 đặc trưng miền TMĐT và biến đổi Log1p y = ln(1 + Amount).
> - Phân chia Train/Test 80/20 và quét chống rò rỉ dữ liệu tuyệt đối (Zero Data Leakage).
> - Thiết lập dây chuyền ColumnTransformer (StandardScaler + OneHotEncoder) và đóng gói hiện vật.


---
## Thiết lập Môi trường & Nạp Dữ liệu Thô Amazon Sale Report
**Mục tiêu:** Khởi tạo cấu hình đường dẫn thư mục tập trung và nạp 128.975 dòng dữ liệu thô ban đầu.


In [2]:
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

BASE_DIR = Path.cwd().parent if Path.cwd().name in ['scripts', 'notebooks'] else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

RAW_CSV = BASE_DIR / 'data' / 'raw' / 'Amazon Sale Report.csv'
CLEAN_CSV = BASE_DIR / 'data' / 'processed' / 'clean_amazon_sales.csv'
df_raw = pd.read_csv(RAW_CSV, low_memory=False)
print(f"✓ Đã nạp thành công dữ liệu thô: {len(df_raw):,} dòng, {df_raw.shape[1]} thuộc tính.")


✓ Đã nạp thành công dữ liệu thô: 128,975 dòng, 24 thuộc tính.


---
## Mục 1: Tiền Xử Lý Dữ Liệu & Khử Trùng Lặp Thô (Data Cleaning & Deduplication)


### 3.1 Xử lý Giá trị Khuyết thiếu & Chuẩn hóa Thuộc tính Định tính
**Mục tiêu:** Điền khuyết thiếu có kiểm soát cho các cột địa lý, khuyến mãi, trạng thái vận chuyển và loại bỏ cột phương sai 0.

In [3]:
from data.scripts.data_preprocessing import handle_missing_and_duplicates

df_filtered = df_raw[~df_raw['Status'].astype(str).str.contains('Cancelled', case=False, na=False)]
df_filtered = df_filtered[(df_filtered['Amount'] > 0) & (df_filtered['Qty'] > 0)]
df_imputed = handle_missing_and_duplicates(df_filtered)
print(f"✓ Hoàn tất xử lý missing: Còn lại {len(df_imputed):,} dòng đơn hàng.")

✓ Hoàn tất xử lý missing: Còn lại 108,071 dòng đơn hàng.


### 3.2 Lọc Đơn Hủy & Khử Trùng lặp Giao dịch (Zero Duplicate Leakage)
**Mục tiêu:** Lọc bỏ các đơn hàng hủy/lỗi và khử trùng lặp giao dịch triệt để bảo đảm nguyên tắc Zero Duplicate Leakage.

In [4]:
from data.scripts.data_preprocessing import clean_amazon_sales_data

CLEAN_CSV = BASE_DIR / 'data' / 'processed' / 'clean_amazon_sales.csv'
df_clean = clean_amazon_sales_data(RAW_CSV, CLEAN_CSV)
print(f"✓ Đã làm sạch toàn diện và xuất {len(df_clean):,} đơn hàng sang: {CLEAN_CSV.name}")

✓ Đã làm sạch toàn diện và xuất 102,852 đơn hàng sang: clean_amazon_sales.csv


### 3.3 Xuất Tệp Dữ liệu Sạch Đã Chuẩn hóa (clean_amazon_sales.csv)
**Mục tiêu:** Đóng gói và lưu trữ tập dữ liệu sạch hoàn thiện sẵn sàng cho giai đoạn tạo đặc trưng.

In [5]:
assert CLEAN_CSV.exists(), "Tệp clean_amazon_sales.csv chưa được tạo thành công!"
df_check = pd.read_csv(CLEAN_CSV, low_memory=False)
print(f"✓ Kiểm định độc lập: Tệp sạch gồm {df_check.shape[0]:,} dòng, {df_check.shape[1]} cột. SẴN SÀNG 100%.")

✓ Kiểm định độc lập: Tệp sạch gồm 102,852 dòng, 12 cột. SẴN SÀNG 100%.


---
## Mục 2: Kỹ Thuật Trích Xuất Đặc Trưng Miền & Biến Đổi Mục Tiêu (Domain Features)


### 4.1 Trích xuất Đặc trưng Chuỗi Thời gian & Nhóm Địa lý
**Mục tiêu:** Bóc tách chuỗi thời gian mùa vụ (`month`, `day_of_week`, `is_weekend`), tạo cờ nhị phân nghiệp vụ (`is_promoted`, `is_b2b`) và gom nhóm Top 10 bang lớn nhất.

In [6]:
from scripts.feature_engineering import extract_domain_features

df_features = extract_domain_features(df_clean)
print(f"✓ Số chiều sau trích xuất đặc trưng miền: {df_features.shape[1]} cột")
print("Các cột mới:", [c for c in df_features.columns if c not in df_clean.columns])

✓ Số chiều sau trích xuất đặc trưng miền: 18 cột
Các cột mới: ['month', 'day_of_week', 'is_weekend', 'is_promoted', 'is_b2b', 'state_grouped']


### 4.2 Biến đổi Log Biến Mục tiêu (Amount)
**Mục tiêu:** Áp dụng biến đổi log1p cho biến Amount nhằm bình thường hóa phân phối và ổn định phương sai cho mô hình hồi quy.

In [7]:
from scripts.feature_engineering import transform_target_log

df_features['Amount_log'] = transform_target_log(df_features['Amount'])
print(f"• Số lượng mẫu đã chuyển đổi: {len(df_features):,}")
print(f"• 5 giá trị log đầu tiên: {np.round(df_features['Amount_log'].head().values, 4)}")

• Số lượng mẫu đã chuyển đổi: 102,852
• 5 giá trị log đầu tiên: [6.0088 5.7991 6.3544 6.7154 6.4831]


### 4.3 Quét & Loại bỏ Đa Cộng Tuyến (|r| > 0.6)
**Mục tiêu:** Đo lường ma trận tương quan Pearson giữa các biến số và loại bỏ các biến dư thừa có hệ số tương quan vượt ngưỡng 0.6.

In [8]:
from scripts.feature_engineering import detect_collinear_features

collinear_pairs = detect_collinear_features(df_features, threshold=0.6)
print(f"• Số cặp đặc trưng có tương quan cao (|r| > 0.6): {len(collinear_pairs)}")
for c1, c2, r in collinear_pairs:
    print(f"   - {c1} & {c2}: r = {r:.4f}")

• Số cặp đặc trưng có tương quan cao (|r| > 0.6): 2
   - day_of_week & is_weekend: r = 0.8006
   - Amount & Amount_log: r = 0.9712


---
## Mục 5: Phân chia Dữ liệu & Lắp ráp Dây chuyền Xử lý (Pipeline)

### 5.1 Phân chia Train & Test 80/20 Chống Rò rỉ Dữ liệu
**Mục tiêu:** Tách tập dữ liệu thành 80% Train và 20% Test trước khi thực hiện bất kỳ phép biến đổi tham số nào.

In [9]:
from scripts.feature_engineering import split_train_test

X_train, X_test, y_train, y_test, y_train_raw, y_test_raw = split_train_test(
    df_features, test_size=0.2, random_state=42
)
print(f"✓ Tập huấn luyện (Train set): {len(X_train):,} mẫu (80%)")
print(f"✓ Tập kiểm thử (Test set):     {len(X_test):,} mẫu (20%)")

✓ Tập huấn luyện (Train set): 82,281 mẫu (80%)
✓ Tập kiểm thử (Test set):     20,571 mẫu (20%)


### 5.2 Xây dựng Dây chuyền Xử lý Tự động (ColumnTransformer)
**Mục tiêu:** Đóng gói bộ biến đổi ColumnTransformer gồm StandardScaler cho biến số và OneHotEncoder cho biến định tính.

In [10]:
from scripts.feature_engineering import build_column_transformer

num_cols = ['Qty', 'month', 'day_of_week', 'is_promoted', 'is_b2b']
cat_cols = ['Category', 'Size', 'Fulfilment', 'ship-service-level', 'Courier Status', 'state_grouped']

preprocessor = build_column_transformer(num_cols, cat_cols)
preprocessor.fit(X_train)

X_train_trans = preprocessor.transform(X_train)
print(f"✓ Không gian đặc trưng sau ColumnTransformer: {X_train_trans.shape[1]} chiều.")

✓ Không gian đặc trưng sau ColumnTransformer: 42 chiều.


### 5.3 Lưu trữ Dữ liệu Phân chia & Dây chuyền Tiền Xử lý
**Mục tiêu:** Xuất dữ liệu train/test ra file CSV trực quan và lưu trữ bundle nhị phân train_test_data.joblib.

In [11]:
from scripts.feature_engineering import export_train_test_csv

PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
SAVED_MODELS_DIR = BASE_DIR / 'models' / 'saved_models'
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

export_train_test_csv(X_train, X_test, y_train_raw, y_test_raw, output_dir=PROCESSED_DIR)

bundle = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'y_train_raw': y_train_raw,
    'y_test_raw': y_test_raw,
    'num_cols': num_cols,
    'cat_cols': cat_cols
}
joblib.dump(bundle, PROCESSED_DIR / 'train_test_data.joblib')
joblib.dump(preprocessor, SAVED_MODELS_DIR / 'preprocessor.joblib')
print(f"✓ Đã lưu trữ thành công train_test_data.joblib và preprocessor.joblib.")

✓ Đã lưu trữ thành công train_test_data.joblib và preprocessor.joblib.
